# Batch TDA on 24 similarity-matrix groups

This notebook reads the 24 similarity-matrix files generated from the 15-coin embeddings,
runs VR-based persistent homology, and saves one CSV per group.

For consistency with the previous workflow in `TDA_results_merged.csv`, this notebook:
- computes Betti-0 and Betti-1 curves on a 50-point epsilon grid;
- stores the epsilon-averaged `betti0` and `betti1` as scalar time-series columns;
- stores `persistent_entropy` from the H1 persistence diagram.


In [1]:
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
from ripser import ripser

SIMILARITY_ROOT = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_similarity_matrices')
OUTPUT_DIR = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_tda_csv')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

similarity_files = sorted(SIMILARITY_ROOT.glob('wasserstein/*.npz')) + sorted(SIMILARITY_ROOT.glob('dtw/*.npz'))

print(f'Found {len(similarity_files)} similarity files.')
print(f'Output directory: {OUTPUT_DIR}')
similarity_files[:3]


Found 24 similarity files.
Output directory: /Users/jane/Documents/202511吾-Systems/8. New data/15coin_tda_csv


[PosixPath('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_similarity_matrices/wasserstein/cmc15_log_return_wasserstein_similarity_m07_tau1.npz'),
 PosixPath('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_similarity_matrices/wasserstein/cmc15_log_return_wasserstein_similarity_m07_tau2.npz'),
 PosixPath('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_similarity_matrices/wasserstein/cmc15_log_return_wasserstein_similarity_m07_tau3.npz')]

In [2]:
pattern = re.compile(r'(wasserstein|dtw)_similarity_m(\d+)_tau(\d+)')


def parse_info(path):
    match = pattern.search(path.stem)
    if not match:
        raise ValueError(f'Cannot parse method/window_size/lag from: {path.name}')
    method = match.group(1)
    window_size = int(match.group(2))
    lag = int(match.group(3))
    return method, window_size, lag


def betti_numbers(diagrams, thresholds):
    betti_0 = np.zeros(len(thresholds), dtype=float)
    betti_1 = np.zeros(len(thresholds), dtype=float)

    if len(diagrams[0]) > 0:
        births_0 = diagrams[0][:, 0]
        deaths_0 = diagrams[0][:, 1]
        for i, eps in enumerate(thresholds):
            betti_0[i] = np.sum((births_0 <= eps) & (deaths_0 > eps))

    if len(diagrams[1]) > 0:
        births_1 = diagrams[1][:, 0]
        deaths_1 = diagrams[1][:, 1]
        for i, eps in enumerate(thresholds):
            betti_1[i] = np.sum((births_1 <= eps) & (deaths_1 > eps))

    return betti_0, betti_1


def persistent_entropy(diagram):
    if len(diagram) == 0:
        return 0.0

    lifetimes = diagram[:, 1] - diagram[:, 0]
    finite = np.isfinite(lifetimes) & (lifetimes > 0)
    lifetimes = lifetimes[finite]

    if len(lifetimes) == 0:
        return 0.0

    probs = lifetimes / lifetimes.sum()
    return float(-(probs * np.log(probs)).sum())


def build_tda_csv(similarity_path):
    data = np.load(similarity_path, allow_pickle=True)
    similarity_matrices = data['similarity_matrices'].astype(float)
    coins = data['coins'].tolist()
    dates = data['dates'].tolist()
    method, window_size, lag = parse_info(similarity_path)

    epsilons = np.linspace(0, np.nanmax(similarity_matrices), 50)
    n_time = similarity_matrices.shape[0]

    betti0_mean = np.zeros(n_time, dtype=float)
    betti1_mean = np.zeros(n_time, dtype=float)
    entropy_series = np.zeros(n_time, dtype=float)

    for t in range(n_time):
        dist_mat = similarity_matrices[t]
        dist_mat = np.nan_to_num(dist_mat)
        dist_mat = (dist_mat + dist_mat.T) / 2.0
        np.fill_diagonal(dist_mat, 0.0)

        result = ripser(dist_mat, distance_matrix=True, maxdim=1)
        diagrams = result['dgms']

        b0_curve, b1_curve = betti_numbers(diagrams, epsilons)
        betti0_mean[t] = b0_curve.mean()
        betti1_mean[t] = b1_curve.mean()
        entropy_series[t] = persistent_entropy(diagrams[1])

    df = pd.DataFrame({
        'time_index': np.arange(n_time),
        'date': dates,
        'betti0': betti0_mean,
        'betti1': betti1_mean,
        'persistent_entropy': entropy_series,
        'method': method,
        'window_size': window_size,
        'lag': lag,
        'n_coins': len(coins),
        'value_type': data['value_type'].tolist() if hasattr(data['value_type'], 'tolist') else data['value_type']
    })

    output_file = OUTPUT_DIR / f'cmc15_tda_{method}_m{window_size:02d}_tau{lag}.csv'
    df.to_csv(output_file, index=False)

    return {
        'input_file': similarity_path.name,
        'output_file': output_file.name,
        'method': method,
        'window_size': window_size,
        'lag': lag,
        'n_time': n_time,
        'n_coins': len(coins),
        'first_date': dates[0],
        'last_date': dates[-1],
        'max_epsilon': float(epsilons[-1])
    }


In [3]:
manifest_rows = []

for similarity_path in similarity_files:
    start = time.time()
    row = build_tda_csv(similarity_path)
    row['runtime_seconds'] = round(time.time() - start, 4)
    manifest_rows.append(row)
    print(
        f"Saved {row['output_file']} | method={row['method']} | "
        f"m={row['window_size']} tau={row['lag']} | n_time={row['n_time']} | {row['runtime_seconds']:.2f}s"
    )

manifest_df = pd.DataFrame(manifest_rows).sort_values(['method', 'window_size', 'lag']).reset_index(drop=True)
manifest_path = OUTPUT_DIR / 'cmc15_tda_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)

print(f'\nManifest saved to: {manifest_path}')
manifest_df


Saved cmc15_tda_wasserstein_m07_tau1.csv | method=wasserstein | m=7 tau=1 | n_time=1867 | 0.51s


Saved cmc15_tda_wasserstein_m07_tau2.csv | method=wasserstein | m=7 tau=2 | n_time=1861 | 0.46s


Saved cmc15_tda_wasserstein_m07_tau3.csv | method=wasserstein | m=7 tau=3 | n_time=1855 | 0.47s


Saved cmc15_tda_wasserstein_m14_tau1.csv | method=wasserstein | m=14 tau=1 | n_time=1860 | 0.47s


Saved cmc15_tda_wasserstein_m14_tau2.csv | method=wasserstein | m=14 tau=2 | n_time=1847 | 0.44s


...

Saved cmc15_tda_dtw_m21_tau3.csv | method=dtw | m=21 tau=3 | n_time=1813 | 0.43s


Saved cmc15_tda_dtw_m28_tau1.csv | method=dtw | m=28 tau=1 | n_time=1846 | 0.43s


Saved cmc15_tda_dtw_m28_tau2.csv | method=dtw | m=28 tau=2 | n_time=1819 | 0.43s


Saved cmc15_tda_dtw_m28_tau3.csv | method=dtw | m=28 tau=3 | n_time=1792 | 0.43s

Manifest saved to: /Users/jane/Documents/202511吾-Systems/8. New data/15coin_tda_csv/cmc15_tda_manifest.csv


,input_file,output_file,method,window_size,lag,n_time,n_coins,first_date,last_date,max_epsilon,runtime_seconds
0,cmc15_log_return_dtw_similarity_m07_tau1.npz,cmc15_tda_dtw_m07_tau1.csv,dtw,7,1,1867,15,2020-08-17,2025-09-26,3.005419,0.4528
1,cmc15_log_return_dtw_similarity_m07_tau2.npz,cmc15_tda_dtw_m07_tau2.csv,dtw,7,2,1861,15,2020-08-23,2025-09-26,3.121439,0.4681
2,cmc15_log_return_dtw_similarity_m07_tau3.npz,cmc15_tda_dtw_m07_tau3.csv,dtw,7,3,1855,15,2020-08-29,2025-09-26,2.265797,0.4654
3,cmc15_log_return_dtw_similarity_m14_tau1.npz,cmc15_tda_dtw_m14_tau1.csv,dtw,14,1,1860,15,2020-08-24,2025-09-26,4.287884,0.4443
4,cmc15_log_return_dtw_similarity_m14_tau2.npz,cmc15_tda_dtw_m14_tau2.csv,dtw,14,2,1847,15,2020-09-06,2025-09-26,3.414202,0.4558
5,cmc15_log_return_dtw_similarity_m14_tau3.npz,cmc15_tda_dtw_m14_tau3.csv,dtw,14,3,1834,15,2020-09-19,2025-09-26,2.922890,0.4508
6,cmc15_log_return_dtw_similarity_m21_tau1.npz,cmc15_tda_dtw_m21_tau1.csv,dtw,21,1,1853,15,2020-08-31,2025-09-26,4.641031,0.4394
7,cmc15_log_return_dtw_similarity_m21_tau2.npz,cmc15_tda_dtw_m21_tau2.csv,dtw,21,2,1833,15,2020-09-20,2025-09-26,4.328282,0.4482
8,cmc15_log_return_dtw_similarity_m21_tau3.npz,cmc15_tda_dtw_m21_tau3.csv,dtw,21,3,1813,15,2020-10-10,2025-09-26,3.383690,0.4348
9,cmc15_log_return_dtw_similarity_m28_tau1.npz,cmc15_tda_dtw_m28_tau1.csv,dtw,28,1,1846,15,2020-09-07,2025-09-26,5.321886,0.4279


In [4]:
sample_path = OUTPUT_DIR / 'cmc15_tda_wasserstein_m21_tau1.csv'
sample_df = pd.read_csv(sample_path)
print(sample_df.columns.tolist())
sample_df.head()


['time_index', 'date', 'betti0', 'betti1', 'persistent_entropy', 'method', 'window_size', 'lag', 'n_coins', 'value_type']


,time_index,date,betti0,betti1,persistent_entropy,method,window_size,lag,n_coins,value_type
0,0,2020-08-31,1.74,0.02,0.488415,wasserstein,21,1,15,log_return
1,1,2020-09-01,1.80,0.00,0.463764,wasserstein,21,1,15,log_return
2,2,2020-09-02,1.84,0.02,0.395636,wasserstein,21,1,15,log_return
3,3,2020-09-03,2.00,0.02,-0.000000,wasserstein,21,1,15,log_return
4,4,2020-09-04,1.90,0.00,0.000000,wasserstein,21,1,15,log_return
